# The Shadow Pandemic: Impact of COVID-19 Lockdowns on Domestic Violence in Europe
### Reproducible analysis notebook

**Author:** Matheus Zardini | Higher Diploma in Data Analytics, 2026

This notebook reproduces the full analysis pipeline used in the project report:
1. Data cleaning & integration (Eurostat `crim_hom_vrel`, `crim_hom_soff`, OxCGRT)
2. Exploratory Data Analysis & visualisations
3. Hypothesis testing (Wilcoxon signed-rank, Mann-Whitney U)
4. OLS regression (lockdown stringency vs. change in DV homicide rate)
5. Machine Learning model (Random Forest Regressor) with Leave-One-Country-Out cross-validation

**Folder layout expected (already set up in this delivery):**
```
ShadowPandemic_Project/
    data/                 <- raw source CSVs (Eurostat x2, OxCGRT)
    notebooks/             <- this notebook
    outputs/data/          <- processed panel + statistical results (auto-generated)
    outputs/figures/        <- all figures (auto-generated)
```
Run all cells top to bottom from within the `notebooks/` folder.


## 1. Data Cleaning & Integration Pipeline

In [1]:
"""
The Shadow Pandemic - Data Cleaning & Integration Pipeline
Loads Eurostat crim_hom_vrel, crim_hom_soff and OxCGRT datasets, filters to
domestic-violence-relevant categories, aggregates OxCGRT to annual country
level, and merges everything into a single analysis-ready panel dataset.
"""
import pandas as pd
import numpy as np

DATA_DIR = "../data"
OUT_DIR = "../outputs/data"

ISO2_TO_ISO3 = {
    "AL": "ALB", "AT": "AUT", "BA": "BIH", "BE": "BEL", "BG": "BGR", "CH": "CHE",
    "CY": "CYP", "CZ": "CZE", "DE": "DEU", "DK": "DNK", "EE": "EST", "EL": "GRC",
    "ES": "ESP", "FI": "FIN", "FR": "FRA", "HR": "HRV", "HU": "HUN", "IE": "IRL",
    "IS": "ISL", "IT": "ITA", "LI": "LIE", "LT": "LTU", "LU": "LUX", "LV": "LVA",
    "ME": "MNE", "MK": "MKD", "MT": "MLT", "NL": "NLD", "NO": "NOR", "PL": "POL",
    "PT": "PRT", "RO": "ROU", "RS": "SRB", "SE": "SWE", "SI": "SVN", "SK": "SVK",
    "TR": "TUR", "XK": "XKX",
}
EXCLUDE_GEO = {"EU_V", "UKC-L", "UKM", "UKN"}


def load_vrel():
    df = pd.read_csv(f"{DATA_DIR}/crim_hom_vrel_linear_2_0.csv")
    df = df[~df["geo"].isin(EXCLUDE_GEO)]
    sub = df[(df["pers_cat"] == "IPTN_FAM") & (df["sex"] == "F") & (df["unit"] == "P_HTHAB")]
    sub = sub[["geo", "TIME_PERIOD", "OBS_VALUE"]].rename(
        columns={"geo": "iso2", "TIME_PERIOD": "year", "OBS_VALUE": "dv_homicide_rate_f"}
    )
    return sub.dropna(subset=["dv_homicide_rate_f"])


def load_soff():
    df = pd.read_csv(f"{DATA_DIR}/crim_hom_soff_linear_2_0.csv")
    df = df[~df["geo"].isin(EXCLUDE_GEO)]
    sub = df[(df["leg_stat"] == "PER_VICT") & (df["sex"] == "F") & (df["unit"] == "P_HTHAB")]
    piv = sub.pivot_table(index=["geo", "TIME_PERIOD"], columns="iccs", values="OBS_VALUE").reset_index()
    piv = piv.rename(columns={
        "geo": "iso2", "TIME_PERIOD": "year",
        "ICCS0101": "homicide_rate_f_total",
        "ICCS03011": "rape_rate_f",
        "ICCS03012": "sexual_violence_rate_f",
    })
    return piv


def load_oxcgrt_annual():
    cols = ["CountryCode", "Jurisdiction", "Date", "StringencyIndex_Average",
            "GovernmentResponseIndex_Average", "ContainmentHealthIndex_Average",
            "EconomicSupportIndex", "C6M_Stay at home requirements"]
    df = pd.read_csv(f"{DATA_DIR}/OxCGRT_nat_latest.csv", usecols=cols)
    df = df[df["Jurisdiction"] == "NAT_TOTAL"].copy()
    df["year"] = df["Date"].astype(str).str[:4].astype(int)
    agg = df.groupby(["CountryCode", "year"]).agg(
        stringency_avg=("StringencyIndex_Average", "mean"),
        gov_response_avg=("GovernmentResponseIndex_Average", "mean"),
        containment_health_avg=("ContainmentHealthIndex_Average", "mean"),
        economic_support_avg=("EconomicSupportIndex", "mean"),
        stay_home_avg=("C6M_Stay at home requirements", "mean"),
    ).reset_index()
    return agg.rename(columns={"CountryCode": "iso3"})


def build_panel():
    vrel = load_vrel()
    soff = load_soff()
    ox = load_oxcgrt_annual()

    panel = pd.merge(vrel, soff, on=["iso2", "year"], how="left")
    panel["iso3"] = panel["iso2"].map(ISO2_TO_ISO3)
    panel = pd.merge(panel, ox, on=["iso3", "year"], how="left")

    pre_years = panel["year"] < 2020
    panel.loc[pre_years, ["stringency_avg", "gov_response_avg",
                           "containment_health_avg", "economic_support_avg",
                           "stay_home_avg"]] = panel.loc[pre_years, ["stringency_avg",
                           "gov_response_avg", "containment_health_avg",
                           "economic_support_avg", "stay_home_avg"]].fillna(0)

    country_names = {
        "AL": "Albania", "AT": "Austria", "BA": "Bosnia and Herzegovina", "BE": "Belgium",
        "BG": "Bulgaria", "CH": "Switzerland", "CY": "Cyprus", "CZ": "Czechia", "DE": "Germany",
        "DK": "Denmark", "EE": "Estonia", "EL": "Greece", "ES": "Spain", "FI": "Finland",
        "FR": "France", "HR": "Croatia", "HU": "Hungary", "IE": "Ireland", "IS": "Iceland",
        "IT": "Italy", "LI": "Liechtenstein", "LT": "Lithuania", "LU": "Luxembourg",
        "LV": "Latvia", "ME": "Montenegro", "MK": "North Macedonia", "MT": "Malta",
        "NL": "Netherlands", "NO": "Norway", "PL": "Poland", "PT": "Portugal", "RO": "Romania",
        "RS": "Serbia", "SE": "Sweden", "SI": "Slovenia", "SK": "Slovakia", "TR": "Turkey",
    }
    panel["country"] = panel["iso2"].map(country_names)

    def period(y):
        if 2018 <= y <= 2019:
            return "pre-pandemic"
        if 2020 <= y <= 2022:
            return "pandemic"
        return "other"
    panel["period"] = panel["year"].apply(period)

    panel = panel.sort_values(["iso2", "year"]).reset_index(drop=True)
    return panel


panel = build_panel()
panel.to_csv(f"{OUT_DIR}/analysis_panel.csv", index=False)
print("Panel shape:", panel.shape)
print("Countries:", panel['iso2'].nunique())
print("Year range:", panel['year'].min(), "-", panel['year'].max())
panel.head(15)


Panel shape: (227, 14)
Countries: 26
Year range: 2015 - 2024


,iso2,year,dv_homicide_rate_f,homicide_rate_f_total,rape_rate_f,sexual_violence_rate_f,iso3,stringency_avg,gov_response_avg,containment_health_avg,economic_support_avg,stay_home_avg,country,period
0,AL,2015,0.70,1.12,4.07,4.28,ALB,0.000000,0.000000,0.000000,0.000000,0.000000,Albania,other
1,AL,2016,0.63,1.20,3.03,4.59,ALB,0.000000,0.000000,0.000000,0.000000,0.000000,Albania,other
2,AL,2017,0.49,0.84,2.81,4.36,ALB,0.000000,0.000000,0.000000,0.000000,0.000000,Albania,other
3,AL,2018,0.80,1.05,2.44,3.98,ALB,0.000000,0.000000,0.000000,0.000000,0.000000,Albania,pre-pandemic
4,AL,2019,0.59,0.91,4.26,4.54,ALB,0.000000,0.000000,0.000000,0.000000,0.000000,Albania,pre-pandemic
5,AT,2015,0.32,0.39,23.85,15.70,AUT,0.000000,0.000000,0.000000,0.000000,0.000000,Austria,other
6,AT,2016,0.52,0.70,25.36,46.32,AUT,0.000000,0.000000,0.000000,0.000000,0.000000,Austria,other
7,AT,2017,0.44,0.83,31.12,54.84,AUT,0.000000,0.000000,0.000000,0.000000,0.000000,Austria,other
8,AT,2018,0.43,0.98,35.60,55.96,AUT,0.000000,0.000000,0.000000,0.000000,0.000000,Austria,pre-pandemic
9,AT,2019,0.35,0.89,35.45,49.60,AUT,0.000000,0.000000,0.000000,0.000000,0.000000,Austria,pre-pandemic


## 2. Exploratory Data Analysis & Visualisations

In [2]:
"""EDA & visualisations for the Shadow Pandemic project."""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

DATA_DIR = "../outputs/data"
FIG_DIR = "../outputs/figures"

panel = pd.read_csv(f"{DATA_DIR}/analysis_panel.csv")

desc = panel.groupby("period")["dv_homicide_rate_f"].agg(["count", "mean", "median", "std", "min", "max"])
desc.to_csv(f"{DATA_DIR}/desc_stats_by_period.csv")
print("Descriptive stats by period:")
print(desc)

eu_desc = panel[panel.year.between(2015, 2024)].groupby("year")["dv_homicide_rate_f"].agg(["mean", "median", "std"])
eu_desc.to_csv(f"{DATA_DIR}/desc_stats_by_year.csv")
print("EU-wide average DV homicide rate by year:")
print(eu_desc)

fig, ax = plt.subplots(figsize=(10, 6))
yearly = panel.groupby("year")["dv_homicide_rate_f"].mean()
ax.plot(yearly.index, yearly.values, marker="o", linewidth=2, color="#c0392b")
ax.axvspan(2020, 2022, color="grey", alpha=0.15, label="Pandemic period (2020-2022)")
ax.set_xlabel("Year")
ax.set_ylabel("Mean DV homicide rate\n(female victims, per 100,000 population)")
ax.set_title("Average Intimate-Partner/Family Homicide Rate of Women\nAcross European Countries, 2015-2024")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig1_timeseries.png", dpi=200)
plt.close()

piv = panel.pivot_table(index="country", columns="year", values="dv_homicide_rate_f")
piv = piv.loc[piv.mean(axis=1).sort_values(ascending=False).index]
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(piv, cmap="Reds", linewidths=0.4, linecolor="white", vmax=2.0,
            cbar_kws={"label": "DV homicide rate per 100,000 (capped at 2.0 for readability)"}, ax=ax)
ax.set_title("Intimate-Partner/Family Homicide Rate of Women by Country and Year\n(micro-states e.g. Liechtenstein show extreme swings from very low case counts)")
ax.set_xlabel("Year")
ax.set_ylabel("Country")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig2_heatmap.png", dpi=200)
plt.close()

pand = panel[panel.period == "pandemic"].dropna(subset=["stringency_avg", "dv_homicide_rate_f"])
fig, ax = plt.subplots(figsize=(9, 7))
sns.regplot(data=pand, x="stringency_avg", y="dv_homicide_rate_f", ax=ax,
            scatter_kws={"alpha": 0.6, "s": 60}, line_kws={"color": "#c0392b"})
ax.set_xlabel("Mean Annual Government Stringency Index (0-100)")
ax.set_ylabel("DV homicide rate (female, per 100,000)")
ax.set_title("Lockdown Stringency vs. Domestic-Violence Homicide Rate\n(EU/EEA countries, 2020-2022)")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig3_scatter_stringency.png", dpi=200)
plt.close()

box_df = panel[panel.period.isin(["pre-pandemic", "pandemic"])]
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=box_df, x="period", y="dv_homicide_rate_f", hue="period",
            order=["pre-pandemic", "pandemic"], palette=["#2980b9", "#c0392b"],
            legend=False, ax=ax)
sns.stripplot(data=box_df, x="period", y="dv_homicide_rate_f", order=["pre-pandemic", "pandemic"],
              color="black", alpha=0.4, size=4, ax=ax)
ax.set_xlabel("")
ax.set_ylabel("DV homicide rate (female, per 100,000)")
ax.set_title("Distribution of DV Homicide Rates:\nPre-pandemic (2018-19) vs Pandemic (2020-22)")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig4_boxplot_period.png", dpi=200)
plt.close()

subset_countries = ["Germany", "France", "Italy", "Spain", "Poland", "Sweden",
                     "Austria", "Netherlands", "Greece", "Hungary", "Ireland", "Croatia"]
sub = panel[panel.country.isin(subset_countries)]
g = sns.relplot(data=sub, x="year", y="dv_homicide_rate_f", col="country", col_wrap=4,
                 kind="line", marker="o", height=2.6, color="#c0392b", facet_kws={"sharey": False})
for ax in g.axes.flat:
    ax.axvspan(2020, 2022, color="grey", alpha=0.15)
g.set_titles("{col_name}")
g.set_axis_labels("Year", "Rate /100k")
g.fig.suptitle("DV Homicide Rate Trajectories by Country", y=1.03)
g.savefig(f"{FIG_DIR}/fig5_small_multiples.png", dpi=200, bbox_inches="tight")
plt.close()

corr_vars = ["dv_homicide_rate_f", "homicide_rate_f_total", "rape_rate_f",
             "sexual_violence_rate_f", "stringency_avg", "gov_response_avg",
             "containment_health_avg", "economic_support_avg", "stay_home_avg"]
corr = panel[panel.period == "pandemic"][corr_vars].corr()
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, vmin=-1, vmax=1)
ax.set_title("Correlation Matrix of Violence Indicators and\nOxCGRT Policy Indices (2020-2022)")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig6_correlation_heatmap.png", dpi=200)
plt.close()

print("All figures saved to", FIG_DIR)


Descriptive stats by period:
              count      mean  median       std  min   max
period                                                    
other           113  0.468496   0.390  0.344742  0.0  2.14
pandemic         68  0.637794   0.455  0.844476  0.0  5.12
pre-pandemic     46  0.440000   0.345  0.326292  0.0  1.61
EU-wide average DV homicide rate by year:
          mean  median       std
year                            
2015  0.462273   0.385  0.310083
2016  0.533182   0.495  0.346140
2017  0.353750   0.335  0.278955
2018  0.437273   0.330  0.275287
2019  0.442500   0.350  0.372958
2020  0.765455   0.505  1.019149
2021  0.623478   0.420  1.006527
2022  0.530000   0.450  0.372327
2023  0.433478   0.360  0.313972
2024  0.571818   0.410  0.444283


All figures saved to ../outputs/figures


## 3. Hypothesis Testing

In [3]:
"""Hypothesis testing: pre-pandemic (2018-19) vs pandemic (2020-22) DV rates."""
import pandas as pd
import numpy as np
from scipy import stats

DATA_DIR = "../outputs/data"
panel = pd.read_csv(f"{DATA_DIR}/analysis_panel.csv")

def paired_country_table(value_col):
    pre = panel[panel.period == "pre-pandemic"].groupby("iso2")[value_col].mean()
    pan = panel[panel.period == "pandemic"].groupby("iso2")[value_col].mean()
    both = pd.concat([pre.rename("pre"), pan.rename("pandemic")], axis=1).dropna()
    return both

results = {}

# ---------------------------------------------------------------------------
# Test 1: Wilcoxon signed-rank test on DV homicide rate (intimate partner/family)
# ---------------------------------------------------------------------------
t1 = paired_country_table("dv_homicide_rate_f")
w1 = stats.wilcoxon(t1["pandemic"], t1["pre"], alternative="greater")
results["dv_homicide_wilcoxon"] = {
    "n_countries": len(t1), "pre_mean": t1["pre"].mean(), "pandemic_mean": t1["pandemic"].mean(),
    "pre_median": t1["pre"].median(), "pandemic_median": t1["pandemic"].median(),
    "statistic": w1.statistic, "p_value": w1.pvalue,
    "n_increased": (t1["pandemic"] > t1["pre"]).sum(),
}

# ---------------------------------------------------------------------------
# Test 2: Wilcoxon signed-rank test on sexual violence rate (secondary indicator)
# ---------------------------------------------------------------------------
t2 = paired_country_table("sexual_violence_rate_f")
w2 = stats.wilcoxon(t2["pandemic"], t2["pre"], alternative="greater")
results["sexual_violence_wilcoxon"] = {
    "n_countries": len(t2), "pre_mean": t2["pre"].mean(), "pandemic_mean": t2["pandemic"].mean(),
    "pre_median": t2["pre"].median(), "pandemic_median": t2["pandemic"].median(),
    "statistic": w2.statistic, "p_value": w2.pvalue,
    "n_increased": (t2["pandemic"] > t2["pre"]).sum(),
}

# ---------------------------------------------------------------------------
# Test 3 (robustness): Mann-Whitney U (independent samples) on total homicide
# rate, all country-year observations rather than country means
# ---------------------------------------------------------------------------
g1 = panel[panel.period == "pre-pandemic"]["dv_homicide_rate_f"].dropna()
g2 = panel[panel.period == "pandemic"]["dv_homicide_rate_f"].dropna()
mw = stats.mannwhitneyu(g2, g1, alternative="greater")
results["dv_homicide_mannwhitney_obslevel"] = {
    "n_pre_obs": len(g1), "n_pandemic_obs": len(g2),
    "statistic": mw.statistic, "p_value": mw.pvalue,
}

# ---------------------------------------------------------------------------
# Effect size: matched-pairs rank-biserial correlation for Wilcoxon test 1
# ---------------------------------------------------------------------------
diffs = t1["pandemic"] - t1["pre"]
n = len(diffs)
r_rb = w1.statistic / (n * (n + 1) / 2) * 2 - 1  # rank-biserial approx from W statistic
results["dv_homicide_wilcoxon"]["effect_size_rank_biserial_approx"] = r_rb

for k, v in results.items():
    print(f"\n=== {k} ===")
    for kk, vv in v.items():
        print(f"  {kk}: {vv}")

pd.DataFrame(results).T.to_csv(f"{DATA_DIR}/hypothesis_test_results.csv")
t1.to_csv(f"{DATA_DIR}/paired_dv_homicide_pre_vs_pandemic.csv")



=== dv_homicide_wilcoxon ===
  n_countries: 22
  pre_mean: 0.43227272727272736
  pandemic_mean: 0.5018181818181819
  pre_median: 0.33999999999999997
  pandemic_median: 0.42500000000000004
  statistic: 194.5
  p_value: 0.013628772096566094
  n_increased: 17
  effect_size_rank_biserial_approx: 0.5375494071146245

=== sexual_violence_wilcoxon ===
  n_countries: 16
  pre_mean: 38.123125
  pandemic_mean: 39.76416666666667
  pre_median: 22.025
  pandemic_median: 14.338333333333333
  statistic: 90.0
  p_value: 0.1372222900390625
  n_increased: 10

=== dv_homicide_mannwhitney_obslevel ===
  n_pre_obs: 46
  n_pandemic_obs: 68
  statistic: 1937.0
  p_value: 0.015691360170002944


## 4. OLS Regression

In [4]:
"""OLS regression: lockdown stringency vs. change in DV homicide rate."""
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

DATA_DIR = "../outputs/data"
FIG_DIR = "../outputs/figures"
panel = pd.read_csv(f"{DATA_DIR}/analysis_panel.csv")

# ---------------------------------------------------------------------------
# Build country-level cross-section: baseline 2019, pandemic-period average
# (2020-2022), and the *variation* (delta) in DV homicide rate.
# ---------------------------------------------------------------------------
base19 = panel[panel.year == 2019].set_index("iso2")["dv_homicide_rate_f"]
pand_mean = panel[panel.period == "pandemic"].groupby("iso2")["dv_homicide_rate_f"].mean()
stringency = panel[panel.period == "pandemic"].groupby("iso2")["stringency_avg"].mean()
gov_resp = panel[panel.period == "pandemic"].groupby("iso2")["gov_response_avg"].mean()
contain = panel[panel.period == "pandemic"].groupby("iso2")["containment_health_avg"].mean()
econ = panel[panel.period == "pandemic"].groupby("iso2")["economic_support_avg"].mean()
stay_home = panel[panel.period == "pandemic"].groupby("iso2")["stay_home_avg"].mean()

cs = pd.concat([base19.rename("baseline_2019"), pand_mean.rename("pandemic_mean"),
                stringency.rename("stringency_avg"), gov_resp.rename("gov_response_avg"),
                contain.rename("containment_health_avg"), econ.rename("economic_support_avg"),
                stay_home.rename("stay_home_avg")], axis=1).dropna()
cs["delta_dv_rate"] = cs["pandemic_mean"] - cs["baseline_2019"]
cs["pct_change_dv_rate"] = 100 * cs["delta_dv_rate"] / cs["baseline_2019"].replace(0, np.nan)
cs.to_csv(f"{DATA_DIR}/country_cross_section.csv")
print("Cross-section n =", len(cs))
print(cs.describe())

# ---------------------------------------------------------------------------
# Model A: simple bivariate OLS  delta_dv_rate ~ stringency_avg
# ---------------------------------------------------------------------------
X = sm.add_constant(cs["stringency_avg"])
y = cs["delta_dv_rate"]
modelA = sm.OLS(y, X).fit()
print("\n===== Model A: delta_dv_rate ~ stringency_avg =====")
print(modelA.summary())

with open(f"{DATA_DIR}/ols_model_A_summary.txt", "w") as f:
    f.write(str(modelA.summary()))

# ---------------------------------------------------------------------------
# Model B: multiple OLS with additional OxCGRT policy indices
# ---------------------------------------------------------------------------
Xb = sm.add_constant(cs[["stringency_avg", "economic_support_avg", "stay_home_avg"]])
modelB = sm.OLS(y, Xb).fit()
print("\n===== Model B: delta_dv_rate ~ stringency + economic_support + stay_home =====")
print(modelB.summary())
with open(f"{DATA_DIR}/ols_model_B_summary.txt", "w") as f:
    f.write(str(modelB.summary()))

# ---------------------------------------------------------------------------
# Diagnostics figure: residuals vs fitted for Model A
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(modelA.fittedvalues, modelA.resid, alpha=0.7, color="#c0392b")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Model A Residuals vs Fitted")

sm.qqplot(modelA.resid, line="45", ax=axes[1])
axes[1].set_title("Model A Q-Q Plot of Residuals")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig7_ols_diagnostics.png", dpi=200)
plt.close()

print("\nModel A R2:", modelA.rsquared, "Adj R2:", modelA.rsquared_adj)
print("Model B R2:", modelB.rsquared, "Adj R2:", modelB.rsquared_adj)


Cross-section n = 20
       baseline_2019  pandemic_mean  stringency_avg  gov_response_avg  \
count      20.000000      20.000000       20.000000         20.000000   
mean        0.396000       0.500000       40.137251         48.090784   
std         0.295393       0.283372        6.510201          7.513573   
min         0.000000       0.046667       30.514233         33.460904   
25%         0.247500       0.365833       35.877998         42.343546   
50%         0.345000       0.425000       39.689028         48.005743   
75%         0.420000       0.566667       44.105418         51.074508   
max         1.450000       1.376667       52.542773         62.542925   

       containment_health_avg  economic_support_avg  stay_home_avg  \
count               20.000000             20.000000      20.000000   
mean                47.182584             54.447262       0.546966   
std                  7.021624             18.223878       0.334253   
min                 35.083781            


Model A R2: 0.027658390069635774 Adj R2: -0.02636058825982901
Model B R2: 0.2112165839870631 Adj R2: 0.0633196934846374


## 5. Machine Learning Model: Random Forest Regressor

In [5]:
"""
Random Forest Regressor for predicting country-year DV homicide rate during
the pandemic period, compared against a linear (OLS-equivalent) baseline
using Leave-One-Country-Out cross-validation (to avoid within-country leakage
across the repeated 2020-2022 observations).
"""
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

DATA_DIR = "../outputs/data"
FIG_DIR = "../outputs/figures"
panel = pd.read_csv(f"{DATA_DIR}/analysis_panel.csv")

# ---------------------------------------------------------------------------
# Build modelling dataset at country-year granularity (pandemic years only)
# ---------------------------------------------------------------------------
baseline = panel[panel.period == "pre-pandemic"].groupby("iso2")["dv_homicide_rate_f"].mean().rename("baseline_rate")
ml_df = panel[panel.period == "pandemic"].merge(baseline, on="iso2", how="left")
feature_cols = ["stringency_avg", "gov_response_avg", "containment_health_avg",
                 "economic_support_avg", "stay_home_avg", "year", "baseline_rate"]
ml_df = ml_df.dropna(subset=feature_cols + ["dv_homicide_rate_f"])
print("ML dataset shape:", ml_df.shape, " | countries:", ml_df["iso2"].nunique())

X = ml_df[feature_cols].values
y = ml_df["dv_homicide_rate_f"].values
groups = ml_df["iso2"].values

logo = LeaveOneGroupOut()

# ---------------------------------------------------------------------------
# Linear baseline (OLS-equivalent) under LOGO-CV
# ---------------------------------------------------------------------------
lin = LinearRegression()
lin_pred = cross_val_predict(lin, X, y, cv=logo, groups=groups)
lin_r2 = r2_score(y, lin_pred)
lin_mae = mean_absolute_error(y, lin_pred)
lin_rmse = mean_squared_error(y, lin_pred) ** 0.5

# ---------------------------------------------------------------------------
# Random Forest Regressor under LOGO-CV
# ---------------------------------------------------------------------------
rf = RandomForestRegressor(n_estimators=500, max_depth=4, min_samples_leaf=3,
                            random_state=42)
rf_pred = cross_val_predict(rf, X, y, cv=logo, groups=groups)
rf_r2 = r2_score(y, rf_pred)
rf_mae = mean_absolute_error(y, rf_pred)
rf_rmse = mean_squared_error(y, rf_pred) ** 0.5

# Naive baseline: always predict each country's own pre-pandemic mean
naive_pred = ml_df["baseline_rate"].values
naive_r2 = r2_score(y, naive_pred)
naive_mae = mean_absolute_error(y, naive_pred)
naive_rmse = mean_squared_error(y, naive_pred) ** 0.5

print("\n=== Leave-One-Country-Out Cross-Validation Results ===")
for name, r2, mae, rmse in [
    ("Naive (predict own pre-pandemic baseline)", naive_r2, naive_mae, naive_rmse),
    ("Linear Regression (OLS-equivalent)", lin_r2, lin_mae, lin_rmse),
    ("Random Forest Regressor", rf_r2, rf_mae, rf_rmse),
]:
    print(f"{name:45s} R2={r2:7.4f}  MAE={mae:7.4f}  RMSE={rmse:7.4f}")

results_df = pd.DataFrame({
    "model": ["Naive baseline", "Linear Regression", "Random Forest"],
    "R2": [naive_r2, lin_r2, rf_r2],
    "MAE": [naive_mae, lin_mae, rf_mae],
    "RMSE": [naive_rmse, lin_rmse, rf_rmse],
})
results_df.to_csv(f"{DATA_DIR}/ml_model_comparison.csv", index=False)

# ---------------------------------------------------------------------------
# Fit RF on full data (for feature importance reporting only -- not for the
# performance numbers above, which come from the unbiased LOGO-CV)
# ---------------------------------------------------------------------------
rf_full = RandomForestRegressor(n_estimators=500, max_depth=4, min_samples_leaf=3,
                                 random_state=42)
rf_full.fit(X, y)
importances = pd.Series(rf_full.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nRandom Forest feature importances (fit on full pandemic-period data):")
print(importances)
importances.to_csv(f"{DATA_DIR}/rf_feature_importances.csv")

fig, ax = plt.subplots(figsize=(9, 6))
importances.sort_values().plot(kind="barh", color="#c0392b", ax=ax)
ax.set_xlabel("Feature importance (Gini importance)")
ax.set_title("Random Forest Feature Importances\n(Predicting DV Homicide Rate, 2020-2022)")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig8_rf_feature_importance.png", dpi=200)
plt.close()

# ---------------------------------------------------------------------------
# Predicted vs actual plot (LOGO-CV predictions) for both models
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), sharex=True, sharey=True)
for ax, pred, title, r2 in [(axes[0], lin_pred, "Linear Regression", lin_r2),
                              (axes[1], rf_pred, "Random Forest", rf_r2)]:
    ax.scatter(y, pred, alpha=0.7, color="#2980b9")
    lims = [min(y.min(), pred.min()) - 0.1, max(y.max(), pred.max()) + 0.1]
    ax.plot(lims, lims, "k--", linewidth=1)
    ax.set_xlabel("Actual DV homicide rate")
    ax.set_ylabel("Predicted (LOGO-CV)")
    ax.set_title(f"{title}\n(LOGO-CV R² = {r2:.3f})")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig9_predicted_vs_actual.png", dpi=200)
plt.close()

print("\nDone.")


ML dataset shape: (59, 15)  | countries: 20



=== Leave-One-Country-Out Cross-Validation Results ===
Naive (predict own pre-pandemic baseline)     R2= 0.3935  MAE= 0.1796  RMSE= 0.2547
Linear Regression (OLS-equivalent)            R2= 0.3783  MAE= 0.1869  RMSE= 0.2579
Random Forest Regressor                       R2= 0.0001  MAE= 0.2329  RMSE= 0.3270



Random Forest feature importances (fit on full pandemic-period data):
baseline_rate             0.676351
containment_health_avg    0.091309
gov_response_avg          0.064214
economic_support_avg      0.061243
stringency_avg            0.058337
stay_home_avg             0.028567
year                      0.019978
dtype: float64



Done.
